## Setup
`add_unknown_member` appends one placeholder row to a dimension, keyed at -1, so any fact row whose natural key doesn't resolve to a real dimension member can be pointed somewhere safe instead of getting a null foreign key.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "transfermarkt"

def silver(t):
    return spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/silver/{t}")

def to_gold(df, name):
    df.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/{name}")

def add_unknown_member(df, key_col, other_defaults):
    row = spark.createDataFrame([(-1,)], [key_col])
    for c, default in other_defaults:
        row = row.withColumn(c, F.lit(default))
    return df.unionByName(row)

players = silver("players")
clubs = silver("clubs")
competitions = silver("competitions")
games = silver("games")
appearances = silver("appearances")
valuations = silver("player_valuations")
transfers = silver("transfers")

## Dim_Date
Built from every date actually used across the four business processes, not a generated calendar range — so it only contains dates that could ever be joined to. date_key uses the yyyyMMdd integer pattern, which is Kimball's standard "smart key" for date dimensions specifically (unlike other dims, a meaningful key is fine here since it's naturally sortable and readable).

In [0]:
dates = (
    games.select("date")
    .union(valuations.select("date"))
    .union(appearances.select("date"))
    .union(transfers.select(F.col("transfer_date").alias("date")))
    .distinct()
    .filter(F.col("date").isNotNull())
    .withColumnRenamed("date", "full_date")
)

dim_date = (
    dates
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
)

dim_date = add_unknown_member(dim_date, "date_key", [
    ("full_date", None), ("year", None), ("quarter", None),
    ("month", None), ("month_name", "Unknown"), ("day", None), ("day_of_week", "Unknown"),
])

to_gold(dim_date, "dim_date")
print("dim_date:", dim_date.count(), "rows")

## Dim_Player
player_key is a warehouse-generated surrogate key via row_number() — fine at this scale (50k rows), though worth knowing that a global row_number() forces everything through one partition to guarantee strict order, which is why monotonically_increasing_id() is the usual choice at true big-data scale instead. Two EDA-driven fixes applied here: international_caps/goals null means zero caps, not unknown, so it's coalesced rather than left null; height_in_cm had a minimum of 17 in the raw data (a clear entry error), so anything outside a plausible human range is nulled out instead of kept.

In [0]:
dim_player = (
    players
    .withColumn("player_key", F.row_number().over(Window.orderBy("player_id")))
    .withColumn("height_in_cm", F.when(F.col("height_in_cm").between(140, 220), F.col("height_in_cm")))
    .select(
        "player_key", "player_id", "name", "first_name", "last_name", "date_of_birth",
        "country_of_birth", "country_of_citizenship",
        "position", "sub_position", "foot", "height_in_cm",
        "current_club_id", "current_club_name",
        F.coalesce("international_caps", F.lit(0)).alias("international_caps"),
        F.coalesce("international_goals", F.lit(0)).alias("international_goals"),
        "market_value_in_eur", "highest_market_value_in_eur",
    )
)

dim_player = add_unknown_member(dim_player, "player_key", [
    ("player_id", None), ("name", "Unknown Player"), ("first_name", None), ("last_name", None),
    ("date_of_birth", None), ("country_of_birth", None), ("country_of_citizenship", None),
    ("position", None), ("sub_position", None), ("foot", None), ("height_in_cm", None),
    ("current_club_id", None), ("current_club_name", None),
    ("international_caps", 0), ("international_goals", 0),
    ("market_value_in_eur", None), ("highest_market_value_in_eur", None),
])

to_gold(dim_player, "dim_player")
print("dim_player:", dim_player.count(), "rows")

## Dim_Club — with inferred members
clubs.csv only covers ~800 "major" clubs, but games/valuations/transfers reference thousands more (youth squads, reserve teams, lower-tier clubs) that were never given a full profile. Rather than collapsing all of them into one generic Unknown row, this recovers a real name for each missing club from wherever it's mentioned — games, valuations, and transfers all carry a name alongside the ID — and inserts it as its own sparse row: only club_id and name populated, everything else null. Anything with no recoverable name still falls back to the -1 Unknown row, same as before. This happens for club IDs that only ever appear via appearances.player_club_id, which carries no name column at all — a real, documented gap, not something worth chasing further.

In [0]:
def club_refs(df, id_col, name_col=None):
    name = F.col(name_col) if name_col else F.lit(None).cast("string")
    return df.select(F.col(id_col).alias("club_id"), name.alias("name"))

all_club_refs = (
    club_refs(games, "home_club_id", "home_club_name")
    .union(club_refs(games, "away_club_id", "away_club_name"))
    .union(club_refs(valuations, "current_club_id", "current_club_name"))
    .union(club_refs(transfers, "from_club_id", "from_club_name"))
    .union(club_refs(transfers, "to_club_id", "to_club_name"))
    .union(club_refs(appearances, "player_club_id"))
    .filter(F.col("club_id").isNotNull())
)

resolved_names = all_club_refs.groupBy("club_id").agg(F.first("name", ignorenulls=True).alias("name"))

# only keep ones we can actually name — nameless ones still fall through to -1 via the coalesce logic already in Cells 7-10
missing_clubs = (
    resolved_names
    .join(clubs.select("club_id"), "club_id", "left_anti")
    .filter(F.col("name").isNotNull())
)

print("Known clubs (clubs.csv):", clubs.count())
print("Inferred members (missing club_id, name recovered elsewhere):", missing_clubs.count())

known_clubs = clubs.select("club_id", "name", "club_code", "domestic_competition_id",
                            "stadium_name", "stadium_seats", "coach_name",
                            "squad_size", "average_age", "total_market_value", "net_transfer_record")

inferred_clubs = missing_clubs.select(
    "club_id", "name",
    F.lit(None).cast("string").alias("club_code"),
    F.lit(None).cast("string").alias("domestic_competition_id"),
    F.lit(None).cast("string").alias("stadium_name"),
    F.lit(None).cast("integer").alias("stadium_seats"),
    F.lit(None).cast("string").alias("coach_name"),
    F.lit(None).cast("integer").alias("squad_size"),
    F.lit(None).cast("double").alias("average_age"),
    F.lit(None).cast("string").alias("total_market_value"),
    F.lit(None).cast("string").alias("net_transfer_record"),
)

dim_club = (
    known_clubs.unionByName(inferred_clubs)
    .withColumn("club_key", F.row_number().over(Window.orderBy("club_id")))
)

dim_club = add_unknown_member(dim_club, "club_key", [
    ("club_id", None), ("name", "Unknown Club"), ("club_code", None),
    ("domestic_competition_id", None), ("stadium_name", None), ("stadium_seats", None),
    ("coach_name", None), ("squad_size", None), ("average_age", None),
    ("total_market_value", None), ("net_transfer_record", None),
])

to_gold(dim_club, "dim_club")
print("dim_club total:", dim_club.count(), "rows")

## Dim_Competition

In [0]:
dim_competition = (
    competitions
    .withColumn("competition_key", F.row_number().over(Window.orderBy("competition_id")))
    .select(
        "competition_key", "competition_id", "name", "type", "sub_type",
        "country_name", "confederation", "domestic_league_code", "total_clubs",
    )
)

dim_competition = add_unknown_member(dim_competition, "competition_key", [
    ("competition_id", None), ("name", "Unknown Competition"), ("type", None),
    ("sub_type", None), ("country_name", None), ("confederation", None),
    ("domestic_league_code", None), ("total_clubs", None),
])

to_gold(dim_competition, "dim_competition")
print("dim_competition:", dim_competition.count(), "rows")

## Load dimension lookups
Read back from what was just written, once, and reused across all four fact tables below — each fact just renames these to whatever role it needs (home/away club, from/to club, etc.).

In [0]:
date_lookup = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/dim_date").select("full_date", "date_key")
club_lookup = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/dim_club").select("club_id", "club_key")
competition_lookup = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/dim_competition").select("competition_id", "competition_key")
player_lookup = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/dim_player").select("player_id", "player_key")

## Fact_Game
Transaction fact, grain: one row per game. Dim_Club is joined twice under different aliases — the role-playing pattern for home vs. away. Every lookup is coalesced to -1 so an unresolved club or competition doesn't produce a null foreign key.

In [0]:
home_club = club_lookup.withColumnRenamed("club_id", "home_club_id").withColumnRenamed("club_key", "home_club_key")
away_club = club_lookup.withColumnRenamed("club_id", "away_club_id").withColumnRenamed("club_key", "away_club_key")

fact_game = (
    games
    .join(date_lookup, games.date == date_lookup.full_date, "left")
    .join(home_club, "home_club_id", "left")
    .join(away_club, "away_club_id", "left")
    .join(competition_lookup, "competition_id", "left")
    .select(
        "game_id",
        F.coalesce("date_key", F.lit(-1)).alias("date_key"),
        F.coalesce("home_club_key", F.lit(-1)).alias("home_club_key"),
        F.coalesce("away_club_key", F.lit(-1)).alias("away_club_key"),
        F.coalesce("competition_key", F.lit(-1)).alias("competition_key"),
        "season", "round", "home_club_goals", "away_club_goals",
        "attendance", "aggregate", "competition_type",
    )
)
to_gold(fact_game, "fact_game")
print("fact_game:", fact_game.count(), "rows")
print("  unresolved home_club:", fact_game.filter(F.col("home_club_key") == -1).count())
print("  unresolved away_club:", fact_game.filter(F.col("away_club_key") == -1).count())

## Fact_Appearance
Transaction fact, grain: one row per player per game — your richest fact table.

In [0]:
appearance_club = club_lookup.withColumnRenamed("club_id", "player_club_id")

fact_appearance = (
    appearances
    .join(date_lookup, appearances.date == date_lookup.full_date, "left")
    .join(player_lookup, "player_id", "left")
    .join(appearance_club, "player_club_id", "left")
    .join(competition_lookup, "competition_id", "left")
    .select(
        "appearance_id", "game_id",
        F.coalesce("date_key", F.lit(-1)).alias("date_key"),
        F.coalesce("player_key", F.lit(-1)).alias("player_key"),
        F.coalesce("club_key", F.lit(-1)).alias("club_key"),
        F.coalesce("competition_key", F.lit(-1)).alias("competition_key"),
        "goals", "assists", "yellow_cards", "red_cards", "minutes_played",
    )
)
to_gold(fact_appearance, "fact_appearance")
print("fact_appearance:", fact_appearance.count(), "rows")

## Fact_PlayerValuation
Periodic snapshot fact, grain: one row per player per valuation date. market_value_in_eur is semi-additive — summable across players at a point in time, not summable across a single player's history.

In [0]:
valuation_club = club_lookup.withColumnRenamed("club_id", "current_club_id")

fact_player_valuation = (
    valuations
    .join(date_lookup, valuations.date == date_lookup.full_date, "left")
    .join(player_lookup, "player_id", "left")
    .join(valuation_club, "current_club_id", "left")
    .select(
        F.coalesce("date_key", F.lit(-1)).alias("date_key"),
        F.coalesce("player_key", F.lit(-1)).alias("player_key"),
        F.coalesce("club_key", F.lit(-1)).alias("club_key"),
        "market_value_in_eur",
    )
)
to_gold(fact_player_valuation, "fact_player_valuation")
print("fact_player_valuation:", fact_player_valuation.count(), "rows")

## Fact_Transfer
Transaction fact, grain: one row per transfer. from_club/to_club is the same role-playing pattern as home/away in Fact_Game. transfer_type carries over the three-way classification validated in the EDA notebook.

In [0]:
from_club = club_lookup.withColumnRenamed("club_id", "from_club_id").withColumnRenamed("club_key", "from_club_key")
to_club = club_lookup.withColumnRenamed("club_id", "to_club_id").withColumnRenamed("club_key", "to_club_key")

def strip_youth_suffix(col):
    return F.trim(F.regexp_replace(col, r"\s+(U1[0-9]|U2[0-9]|II|III|B|Youth|Yth\.?)$", ""))

transfers_typed = (
    transfers
    .withColumn("from_club_root", strip_youth_suffix(F.col("from_club_name")))
    .withColumn("to_club_root", strip_youth_suffix(F.col("to_club_name")))
    .withColumn(
        "transfer_type",
        F.when(F.col("transfer_fee") > 0, "paid")
         .when(F.col("from_club_root") == F.col("to_club_root"), "internal_squad_move")
         .otherwise("free_or_loan")
    )
)

fact_transfer = (
    transfers_typed
    .join(date_lookup, transfers_typed.transfer_date == date_lookup.full_date, "left")
    .join(player_lookup, "player_id", "left")
    .join(from_club, "from_club_id", "left")
    .join(to_club, "to_club_id", "left")
    .select(
        F.coalesce("date_key", F.lit(-1)).alias("date_key"),
        F.coalesce("player_key", F.lit(-1)).alias("player_key"),
        F.coalesce("from_club_key", F.lit(-1)).alias("from_club_key"),
        F.coalesce("to_club_key", F.lit(-1)).alias("to_club_key"),
        "transfer_season", "transfer_fee", "market_value_in_eur", "transfer_type",
    )
)
to_gold(fact_transfer, "fact_transfer")
print("fact_transfer:", fact_transfer.count(), "rows")
fact_transfer.groupBy("transfer_type").count().show()

## Sanity checks
Every fact should have exactly as many rows as its Silver source — the star schema reshapes and enriches, it should never drop or duplicate rows.

In [0]:
print("fact_game row count matches games:", fact_game.count() == games.count())
print("fact_appearance row count matches appearances:", fact_appearance.count() == appearances.count())
print("fact_player_valuation row count matches valuations:", fact_player_valuation.count() == valuations.count())
print("fact_transfer row count matches transfers:", fact_transfer.count() == transfers.count())